TRANSFORMATIONS FOR CUSTOMERS

In [0]:
pip install phonenumbers

In [0]:
from pyspark.sql.functions import *
import utils
import phonenumbers
print ("utils imported successfully")


In [0]:
from utils.constants import *

In [0]:
def read_bronze_customers(spark):
    df = spark.read.format("csv")\
        .option("header",True)\
            .option("inferschema",True)\
                .load(f"{BRONZE}/customers/date=2026-05-16/customers.csv")
    print("loaded successfully")
    return df
df= read_bronze_customers(spark)

def transfrom_bronze_customers(df):
    # cleaning phone numbers 
    df_silver = df.withColumn("phone_clean",regexp_replace("phone","[^0-9]",""))\
                    .withColumn("phone_clean",regexp_replace("phone","x.*",""))\
                    .withColumn("phone_clean",regexp_replace("phone_clean","\\.","-"))\
                    .withColumn("first_name",split(col("name")," ")[0])\
                    .withColumn("last_name",split(col("name")," ")[1])\
                    .withColumn("state",initcap(col("state")))\
                    .drop("city","name","phone")\
                    .orderBy("customer_id","first_name","last_name","email","phone_clean","city","state","customer_segment","registration_date","ingestion_date","pipeline_version")
                   

    
    return df_silver
df_silver = transfrom_bronze_customers(df)

def write_silver_customers(df_silver):
    df_silver.write.mode("overwrite")\
        .format("delta")\
        .save(f"{SILVER}/customers/date=2026-05-16/customers.csv")
    print("written successfully")
    
df_silver_cust=write_silver_customers(df_silver)






In [0]:
def read_data_bronze_orders(spark):
    df=spark.read.format("csv")\
        .option("header",True)\
            .option("inferschema",True)\
                .load(f"{BRONZE}/orders/date=2026-05-16/orders.csv")

    print("loaded successfully")            
    return df
df= read_data_bronze_orders(spark)

def transform_bronze_orders(df):
    df_silver = df.filter(col("total_amount")>0)\
                .dropDuplicates(["order_id"])\
                .withColumn("order_value_category",when(col("total_amount")<=1000,"Small")
                .when(col("total_amount")<=5000,"Medium")
                .when(col("total_amount")>5000,"Large" ))\
                .withColumn("is_cancelled",when(col("status")=="Cancelled","True").otherwise("False"))

    df_silver.display()                        
    return df_silver
df_silver = transform_bronze_orders(df)

def write_silver_orders(df_silver):
    df_silver.write.mode("overwrite")\
        .format("delta")\
        .save(f"{SILVER}/orders/date=2026-05-16/orders.csv")
    print("written successfully")
    
df_silver_orders=write_silver_orders(df_silver)

    
    



In [0]:
def read_raw_data_payments(spark):
    df=spark.read.format("csv")\
        .option("header",True)\
            .option("inferschema",True)\
                .load(f"{BRONZE}/payments/date=2026-05-16/payments.csv")

    print("loaded successfully")            
    return df
df= read_raw_data_payments(spark)


def transform_raw_data_payments(df):
    df_silver = df.filter(col("payment_id").isNotNull())\
                .dropDuplicates(["payment_id"])


    return df_silver
df_silver = transform_raw_data_payments(df)
display(df_silver)

def write_silver_payments(df_silver):
    df_silver.write.mode("overwrite")\
        .format("delta")\
        .save(f"{SILVER}/payments/date=2026-05-16/payments.csv")
    print("written successfully")
    
df_silver_payments=write_silver_payments(df_silver)

                
